In [1]:
# ── Install dependencies (run once) ──────────────────────────
#!pip install torch transformers datasets peft accelerate scikit-learn matplotlib flask requests -q

import warnings
warnings.filterwarnings("ignore")

# Quick environment check
import importlib, sys

required = ["torch", "transformers", "datasets", "peft", "sklearn", "matplotlib", "flask", "requests"]
missing = [pkg for pkg in required if importlib.util.find_spec(pkg.replace("-", "_")) is None]

if missing:
    print(f"\u274c Missing packages: {missing}")
    print("   Run: pip install torch transformers datasets peft accelerate scikit-learn matplotlib flask requests")
else:
    print("\u2705 All packages available")
    import torch
    print(f"   PyTorch {torch.__version__}  |  CUDA: {torch.cuda.is_available()} | MPS: {torch.backends.mps.is_available()}")


import os
import textwrap
import warnings

from dotenv import load_dotenv

warnings.filterwarnings("ignore")



def pretty_print(*args):
    text = " ".join(str(arg) for arg in args)
    try:
        print(textwrap.fill(text, width=80))
    except Exception as e:
        print(text)  # fallback to normal print if text is not a string

        


✅ All packages available
   PyTorch 2.10.0  |  CUDA: False | MPS: True


In [2]:
target_model_name = 'tinyllama:latest'

In [3]:
import requests, json, textwrap

# Create a session that ignores proxy env vars (bypasses VPN for local Ollama)
SESSION = requests.Session()
SESSION.trust_env = False  # equivalent of ollama.Client(..., trust_env=False)


def ollama_list_models():
    r = SESSION.get("http://localhost:11434/api/tags", timeout=30)
    r.raise_for_status()
    data = r.json()
    return [m.get("name", "") for m in (data.get("models") or [])]

def check_ollama_ready():
    models = ollama_list_models()
    print(f"Ollama reachable. {len(models)} model(s) found.")
    if target_model_name not in models:
        print("Warning: chat model tinyllama not found in `ollama list`.")

def ask_ollama(prompt, model=target_model_name, system=None, temperature=0.7):
    """Send a prompt to Ollama and return the response text."""
    url = "http://localhost:11434/api/chat"
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})

    payload = {
        "model": model,
        "messages": messages,
        "stream": False,
        "options": {"temperature": temperature, "num_predict": 256},
    }

    try:
        resp = SESSION.post(url, json=payload, timeout=120)
        resp.raise_for_status()
        data = resp.json()

        # /api/chat returns message.content; keep /api/generate fallback for robustness.
        content = (data.get("message") or {}).get("content")
        if content:
            return content
        if "response" in data:
            return data["response"]
        return f"⚠️ Unexpected Ollama response format: {data}"
    except requests.ConnectionError:
        return "⚠️ Ollama not running. Start it with: ollama serve"
    except Exception as e:
        return f"⚠️ Error: {e}"

def show(label, text, width=90):
    """Pretty-print a response."""
    sep = "=" * width
    print(f"\n{sep}")
    print(f"  {label}")
    print(sep)
    pretty_print(text)
    print()


check_ollama_ready()

Ollama reachable. 10 model(s) found.


In [4]:
# ── A question that demands domain expertise ─────────────────
medical_q = (
    "A 62-year-old male presents with crushing substernal chest pain "
    "radiating to the left arm, diaphoresis, and ST-elevation in leads II, III, and aVF. "
    "What is the most likely diagnosis and immediate management?"
)

response = ask_ollama(medical_q, model=target_model_name)
show("TinyLlama (Base) \u2014 Medical Question", response)


  TinyLlama (Base) — Medical Question
The most likely diagnosis is aortic stenosis with acute on chronic left
ventricular systolic dysfunction due to aortic stenosis. The immediate
management would be to start medications to treat the underlying condition
(e.g., angiotensin-converting enzyme (ACE) inhibitors and beta-blockers) and
perform an echocardiogram to monitor the progression of the disease. In severe
cases, surgical intervention may be necessary to replace the stenotic segment of
the aorta.



In [5]:
# ── Customer support: we want concise, empathetic, branded ──
support_q = (
    "Customer says: I have been on hold for 45 minutes and my order "
    "still has not shipped. This is ridiculous.\n\n"
    "Write a response as a support agent for TechMart Electronics."
)

response = ask_ollama(support_q, model=target_model_name)
show("TinyLlama (Base) \u2014 Customer Support", response)


  TinyLlama (Base) — Customer Support
Customer: I'm so sorry for the inconvenience. I understand your frustration with
the extended wait time on hold.  Agent: Thank you for bringing this to our
attention. We've made some changes to our ordering process and are now
processing orders within the promised time frame. We will have your order
shipped to you as soon as possible.  Customer: That's great news. I appreciate
your prompt response.  Agent: Of course, we're always working to improve our
service and make it as smooth as possible for our customers. Please let us know
if there's anything else we can do for you.  Customer: No need to apologize for
my order. I'm more than happy to do business with you.  Agent: Thank you for
choosing TechMart Electronics. We appreciate your business and look forward to
providing you with high-quality products and exceptional service in the future.
Customer: I hope so. By the way, the wait time on hold is still too long.
Agent: We'll do our best to reduce

In [12]:
# ── Structured output: base models often fumble formats ─────
json_q = (
    "Extract the following from this text and return ONLY valid JSON:\n\n"
    "Text: Dr. Sarah Chen, a cardiologist at Mayo Clinic, published a study on "
    "atrial fibrillation treatment using novel anticoagulants in March 2024.\n\n"
    "Required JSON keys: name, specialty, institution, topic, date. Also I want only and only valid JSON with no extra commentary or text. If any info is missing, use null for that key. AGAIN ONLY VALID JSON WITH NO EXTRA TEXT OR COMMENTARY."
)

response = ask_ollama(json_q, model=target_model_name)
show("TinyLlama (Base) \u2014 Structured Extraction", response)

# Let's see if it's actually valid JSON

try:
    extracted = json.loads(response)
    print("\u2705 Valid JSON extracted:")
    print(json.dumps(extracted, indent=2))
except json.JSONDecodeError as e:
    print(f"\u274c Invalid JSON: {e}")


  TinyLlama (Base) — Structured Extraction
[     {         "name": "Dr. Sarah Chen",         "specialty": "Cardiologist",
"institution": "Mayo Clinic",         "topic": "Atrial Fibrillation Treatment
with Novel AntiCoagulements",         "date": "2024-03-01"     } ]

✅ Valid JSON extracted:
[
  {
    "name": "Dr. Sarah Chen",
    "specialty": "Cardiologist",
    "institution": "Mayo Clinic",
    "topic": "Atrial Fibrillation Treatment with Novel AntiCoagulements",
    "date": "2024-03-01"
  }
]


# Let's use LoRA

In [16]:
# ── Step 1: Load the base model ───────────────────────────────
import truststore
truststore.inject_into_ssl()


from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

MODEL_NAME = "/Users/shivam13juna/Documents/scaler/iitr_classes/llm_ref/distilgpt2"


print(f"Loading {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token  # GPT-2 has no pad token

base_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)

# Count parameters
total_params = sum(p.numel() for p in base_model.parameters())
print(f"\u2705 Loaded: {total_params:,} parameters ({total_params/1e6:.1f}M)")


Loading /Users/shivam13juna/Documents/scaler/iitr_classes/llm_ref/distilgpt2...


Loading weights: 100%|██████████| 76/76 [00:00<00:00, 18665.37it/s]

✅ Loaded: 81,912,576 parameters (81.9M)


In [17]:
# ── Step 2: Test the base model BEFORE fine-tuning ───────────
def generate_text(model, prompt, max_new_tokens=100):
    """Generate text from a model given a prompt."""
    inputs = tokenizer(prompt, return_tensors="pt", padding=True, truncation=True)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.2
        )

    generated = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return generated.strip()

# ── Test with a support-style prompt ─────────────────────────
test_prompt = "Customer: My order #4521 hasn't arrived after 2 weeks.\nAgent:"

print("\U0001f4cb BEFORE Fine-Tuning:")
print(f"   Prompt: {test_prompt}")
print(f"   Response: {generate_text(base_model, test_prompt)}")


📋 BEFORE Fine-Tuning:
   Prompt: Customer: My order #4521 hasn't arrived after 2 weeks.
Agent:
   Response: I'm sorry! Since leaving for the other week, my delivery has been delayed because of a bad customer service from an earlier date (I wasn’t able to get it through). Thank you so much :)


In [18]:

# -- Step 3: Build customer support QA-style pairs --
support_examples = [
    (
        "Customer: My order hasn't arrived yet.\nAgent:",
        " I'm sorry to hear that! Let me look into your order right away. Could you please share your order number so I can track it for you?\n",
    ),
    (
        "Customer: I want a refund for my broken headphones.\nAgent:",
        " I completely understand your frustration. We'll get this sorted out. I'm initiating a refund for you right now.\n",
    ),
    (
        "Customer: How do I reset my password?\nAgent:",
        " Great question! Go to Settings > Account > Reset Password. You'll receive an email with a reset link.\n",
    ),
    (
        "Customer: Your app keeps crashing on my phone.\nAgent:",
        " I'm sorry about that! Let's troubleshoot: first, try updating the app to the latest version. If that doesn't help, clear the app cache.\n",
    ),
    (
        "Customer: I was charged twice for the same item.\nAgent:",
        " Oh no, that shouldn't happen! I can see the duplicate charge. I'm processing a refund for the extra charge right now.\n",
    ),
    (
        "Customer: Can I change my delivery address?\nAgent:",
        " Of course! If your order hasn't shipped yet, I can update the address right away. Could you please provide the new address?\n",
    ),
    (
        "Customer: The product I received is the wrong color.\nAgent:",
        " I apologize for the mix-up! I'll arrange a replacement in the correct color to be shipped out today at no extra cost!\n",
    ),
    (
        "Customer: I need to cancel my subscription.\nAgent:",
        " I understand. I can process that cancellation right now. Your subscription will remain active until the end of your current billing period.\n",
    ),
    (
        "Customer: The website won't accept my coupon code.\nAgent:",
        " Let me check that code for you! Sometimes codes are case-sensitive or have an expiration date. What code are you using?\n",
    ),
    (
        "Customer: I never received my confirmation email.\nAgent:",
        " No worries! Let me resend that for you. Could you confirm the email address on your account?\n",
    ),
    (
        "Customer: My package arrived damaged.\nAgent:",
        " I'm so sorry about that! I'm sending a replacement right away at no charge. You don't need to return the damaged item.\n",
    ),
    (
        "Customer: How long does shipping usually take?\nAgent:",
        " Standard shipping typically takes 5-7 business days. We also offer express (2-3 days) and overnight options at checkout.\n",
    ),
]


In [ ]:
# ── Tokenize the dataset ──────────────────────────────────────
from torch.utils.data import Dataset, DataLoader

class SupportDataset(Dataset):
    def __init__(self, pairs, tokenizer, max_length=256):
        self.items = []

        for prompt, answer in pairs:
            full_text = prompt + answer
            enc = tokenizer(
                full_text,
                truncation=True,
                max_length=max_length,
                padding="max_length",
                return_tensors="pt"
            )

            input_ids = enc["input_ids"].squeeze(0)
            attention_mask = enc["attention_mask"].squeeze(0)

            labels = input_ids.clone()

            prompt_ids = tokenizer(prompt, add_special_tokens=False)["input_ids"]
            prompt_len = len(prompt_ids)

            labels[:prompt_len] = -100 # don't want input to affect my loss
            labels[attention_mask == 0] = -100 # I don't want padding to affect my loss

            self.items.append({
                "input_ids": input_ids,
                "attention_mask": attention_mask,
                "labels": labels
            })

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        return self.items[idx]

dataset = SupportDataset(support_examples, tokenizer)
dataloader = DataLoader(dataset, batch_size=4, shuffle=True)
print(f"\u2705 Dataset ready: {len(dataset)} examples, batch_size=4")


✅ Dataset ready: 12 examples, batch_size=4


In [20]:
# print layers in the model

for name, layer in base_model.named_modules():
    print(name)


transformer
transformer.wte
transformer.wpe
transformer.drop
transformer.h
transformer.h.0
transformer.h.0.ln_1
transformer.h.0.attn
transformer.h.0.attn.c_attn
transformer.h.0.attn.c_proj
transformer.h.0.attn.attn_dropout
transformer.h.0.attn.resid_dropout
transformer.h.0.ln_2
transformer.h.0.mlp
transformer.h.0.mlp.c_fc
transformer.h.0.mlp.c_proj
transformer.h.0.mlp.act
transformer.h.0.mlp.dropout
transformer.h.1
transformer.h.1.ln_1
transformer.h.1.attn
transformer.h.1.attn.c_attn
transformer.h.1.attn.c_proj
transformer.h.1.attn.attn_dropout
transformer.h.1.attn.resid_dropout
transformer.h.1.ln_2
transformer.h.1.mlp
transformer.h.1.mlp.c_fc
transformer.h.1.mlp.c_proj
transformer.h.1.mlp.act
transformer.h.1.mlp.dropout
transformer.h.2
transformer.h.2.ln_1
transformer.h.2.attn
transformer.h.2.attn.c_attn
transformer.h.2.attn.c_proj
transformer.h.2.attn.attn_dropout
transformer.h.2.attn.resid_dropout
transformer.h.2.ln_2
transformer.h.2.mlp
transformer.h.2.mlp.c_fc
transformer.h.2.mlp

In [21]:
# ── Step 4: Apply LoRA adapters ───────────────────────────────
from peft import get_peft_model, LoraConfig, TaskType

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,                    # Rank of the low-rank matrices
    lora_alpha=32,          # Scaling factor
    lora_dropout=0.1,       # Regularization
    target_modules=["c_attn", "c_proj"],  # Which layers get LoRA
)

model = get_peft_model(base_model, lora_config)

# ── The big reveal: parameter comparison ─────────────────────
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
all_params = sum(p.numel() for p in model.parameters())


In [22]:
for name, layer in model.named_modules():
    print(name)


base_model
base_model.model
base_model.model.transformer
base_model.model.transformer.wte
base_model.model.transformer.wpe
base_model.model.transformer.drop
base_model.model.transformer.h
base_model.model.transformer.h.0
base_model.model.transformer.h.0.ln_1
base_model.model.transformer.h.0.attn
base_model.model.transformer.h.0.attn.c_attn
base_model.model.transformer.h.0.attn.c_attn.base_layer
base_model.model.transformer.h.0.attn.c_attn.lora_dropout
base_model.model.transformer.h.0.attn.c_attn.lora_dropout.default
base_model.model.transformer.h.0.attn.c_attn.lora_A
base_model.model.transformer.h.0.attn.c_attn.lora_A.default
base_model.model.transformer.h.0.attn.c_attn.lora_B
base_model.model.transformer.h.0.attn.c_attn.lora_B.default
base_model.model.transformer.h.0.attn.c_attn.lora_embedding_A
base_model.model.transformer.h.0.attn.c_attn.lora_embedding_B
base_model.model.transformer.h.0.attn.c_attn.lora_magnitude_vector
base_model.model.transformer.h.0.attn.c_proj
base_model.model.

In [23]:
# ── Step 5: Training loop ─────────────────────────────────────
from torch.optim import AdamW
import time

optimizer = AdamW(model.parameters(), lr=5e-4)
model.train()

NUM_EPOCHS = 10
losses = []

print("\U0001f3cb\ufe0f Training LoRA adapters...")
print("-" * 50)

start_time = time.time()

for epoch in range(NUM_EPOCHS):
    epoch_loss = 0
    for batch in dataloader:
        optimizer.zero_grad()
        outputs = model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
            labels=batch["labels"]
        )
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()

    avg_loss = epoch_loss / len(dataloader)
    losses.append(avg_loss)

    bar = "\u2588" * int(30 * (epoch + 1) / NUM_EPOCHS) + "\u2591" * (30 - int(30 * (epoch + 1) / NUM_EPOCHS))
    print(f"  Epoch {epoch+1:2d}/{NUM_EPOCHS} |{bar}| Loss: {avg_loss:.4f}")

elapsed = time.time() - start_time
print(f"\n\u2705 Training complete in {elapsed:.1f}s")


🏋️ Training LoRA adapters...
--------------------------------------------------


`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


  Epoch  1/10 |███░░░░░░░░░░░░░░░░░░░░░░░░░░░| Loss: 3.2597
  Epoch  2/10 |██████░░░░░░░░░░░░░░░░░░░░░░░░| Loss: 3.0521
  Epoch  3/10 |█████████░░░░░░░░░░░░░░░░░░░░░| Loss: 2.8529
  Epoch  4/10 |████████████░░░░░░░░░░░░░░░░░░| Loss: 2.6829
  Epoch  5/10 |███████████████░░░░░░░░░░░░░░░| Loss: 2.5337
  Epoch  6/10 |██████████████████░░░░░░░░░░░░| Loss: 2.3291
  Epoch  7/10 |█████████████████████░░░░░░░░░| Loss: 2.2395
  Epoch  8/10 |████████████████████████░░░░░░| Loss: 1.9751
  Epoch  9/10 |███████████████████████████░░░| Loss: 1.8714
  Epoch 10/10 |██████████████████████████████| Loss: 1.7446

✅ Training complete in 15.9s


In [25]:
# ── Compare before and after ──────────────────────────────────
model.eval()

test_prompts = [
    "Customer: My order #4521 hasn't arrived after 2 weeks.\nAgent:",
    "Customer: I want a refund.\nAgent:",
    "Customer: Your app keeps crashing.\nAgent:",
]

from transformers import AutoModelForCausalLM
base_for_comparison = AutoModelForCausalLM.from_pretrained(MODEL_NAME)

print("\U0001f52c Side-by-Side Comparison: Base vs LoRA Fine-Tuned")
print("=" * 80)

for prompt in test_prompts:
    print(f"\n\U0001f4cb Prompt: {prompt}")
    print("-" * 80)

    base_response = generate_text(base_for_comparison, prompt, max_new_tokens=80)
    print(f"  \U0001f7e1 BASE:      {base_response[:200]}")

    ft_response = generate_text(model, prompt, max_new_tokens=80)
    print(f"  \U0001f7e2 FINE-TUNED: {ft_response[:200]}")
    print()


Loading weights: 100%|██████████| 76/76 [00:00<00:00, 12769.07it/s]


🔬 Side-by-Side Comparison: Base vs LoRA Fine-Tuned

📋 Prompt: Customer: My order #4521 hasn't arrived after 2 weeks.
Agent:
--------------------------------------------------------------------------------
  🟡 BASE:      I'm sorry, it looks like we're not paying anything for the food to buy and have no problems with this product!
  🟢 FINE-TUNED: I'm sorry to hear that! Let me check your exchange status for you right now at checkout today! Can anyone please confirm the return address? Are any other orders received with a replacement number or 


📋 Prompt: Customer: I want a refund.
Agent:
--------------------------------------------------------------------------------
  🟡 BASE:      Thanks, Agent!
  🟢 FINE-TUNED: No problem! You can return the current version to your account right now, and we'll see what happens next at that point.</p>
(function() { this.addEventListener('resume-to", function(){ console . log 


📋 Prompt: Customer: Your app keeps crashing.
Agent:
------------------------

In [26]:
# ── Show how tiny the LoRA adapter is ─────────────────────────
import os, tempfile

save_dir = tempfile.mkdtemp()
model.save_pretrained(save_dir)

adapter_size = sum(
    os.path.getsize(os.path.join(save_dir, f)) 
    for f in os.listdir(save_dir)
) / (1024 * 1024)

base_size = sum(p.numel() * p.element_size() for p in base_for_comparison.parameters()) / (1024 * 1024)

print("\U0001f4be Storage Comparison")
print("=" * 50)
print(f"  Base model:    {base_size:>8.1f} MB")
print(f"  LoRA adapter:  {adapter_size:>8.1f} MB")
print(f"  Ratio:         {adapter_size/base_size*100:>7.2f}%")
print("=" * 50)
print(f"\n\U0001f4a1 You can store ~{int(base_size/max(adapter_size,0.01))} different LoRA adapters")
print(f"   for the space of ONE full model copy!")
print(f"   \u2192 Legal adapter, Medical adapter, Support adapter...")
print(f"      all sharing the same base model.")


💾 Storage Comparison
  Base model:       312.5 MB
  LoRA adapter:       1.6 MB
  Ratio:            0.50%

💡 You can store ~200 different LoRA adapters
   for the space of ONE full model copy!
   → Legal adapter, Medical adapter, Support adapter...
      all sharing the same base model.


Link: https://arxiv.org/pdf/2501.00365